# 03 · Modelling & Ablation Study  (CRISP-DM phase 4)

Plan
1. Chronological split **inside every series** (first 70 % train / next 15 % validation / last 15 % test) – no random shuffling, so no future leaks into the past of a series.
2. Baselines: naive (last value) and seasonal-naive.
3. **Screen 8 regression algorithms** on validation MAPE → keep the **5 best**.
4. Light hyper-parameter tuning on validation.
5. **Ablation study** on the 5 models: remove one feature group at a time, and test excluding imputed rows from training.
6. Pick the best (smallest validation MAPE, must be < 10 %), build **prediction intervals** (normalised split-conformal), report the test result, refit on all data and save.

Metrics are always computed on **real** target months only (`y_imputed == 0`).

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
import prep
FIG = os.path.join(ROOT, "reports", "figures"); os.makedirs(FIG, exist_ok=True)
RAW = os.path.join(ROOT, "data", "wfp_food_prices_zmb.csv")
PROC = os.path.join(ROOT, "data", "processed"); os.makedirs(PROC, exist_ok=True)
MODELS = os.path.join(ROOT, "models"); os.makedirs(MODELS, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "figure.autolayout": True})
def save(name):
    plt.savefig(os.path.join(FIG, name), bbox_inches="tight"); plt.show()

import time
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
import lightgbm as lgb, xgboost as xgb
F = pd.read_csv(os.path.join(PROC, "features.csv"), parse_dates=["month","next_month"])
codes = json.load(open(os.path.join(PROC, "codes.json")))
F = F.sort_values(["series_id","month"]).reset_index(drop=True)

## 4.1 Chronological split inside each series

In [ ]:
F["rk"] = F.groupby("series_id").cumcount(); F["n"] = F.groupby("series_id").month.transform("size")
F["split"] = np.where(F.rk >= F.n*0.85, "test", np.where(F.rk >= F.n*0.70, "val", "train"))
print(F.split.value_counts())
fig, ax = plt.subplots(figsize=(9,2.8))
F.groupby(["month","split"]).size().unstack(fill_value=0).plot.area(ax=ax, stacked=True, alpha=.8)
ax.set_title("Rows per month and split (each series is cut chronologically)")
save("03_split_timeline.png")

## 4.2 Metrics and baselines
`MAPE` is the headline metric (target < 10 %). `MAE` is in ZMW per standard unit; `MdAPE` is the median % error.

In [ ]:
def metrics(d, pred_price):
    ok = d.y_imputed.values == 0
    y, p = d.y_price.values[ok], np.asarray(pred_price)[ok]
    ape = np.abs(y-p)/y
    return {"MAPE_%": ape.mean()*100, "MdAPE_%": np.median(ape)*100, "MAE_ZMW": np.abs(y-p).mean(), "n": int(ok.sum())}

tr = F[F.split=="train"]; va = F[F.split=="val"]; te = F[F.split=="test"]
rows = []
for name, fn in {"naive (last value)": lambda d: d.price.values,
                 "seasonal naive (+avg change)": lambda d: d.price.values*np.exp(d.seas_index.fillna(0).values)}.items():
    rows.append({"model": name, **{f"val_{k}":v for k,v in metrics(va, fn(va)).items() if k!="n"},
                 **{f"test_{k}":v for k,v in metrics(te, fn(te)).items()}})
baselines = pd.DataFrame(rows); display(baselines.round(3))

## 4.3 Screening 8 candidate algorithms

Models are trained on the **log price change**. Tree boosters use an **L1 loss** (it matches MAE/MAPE better than squared error and is robust to the occasional price spike). Linear/forest models get a median imputer for missing lags (and scaling for the linear ones); LightGBM/XGBoost/HistGB handle missing values natively.

In [ ]:
def make_model(name, **kw):
    if name == "Ridge":        return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=kw.get("alpha", 10)))
    if name == "ElasticNet":   return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), ElasticNet(alpha=kw.get("alpha", 0.001), l1_ratio=kw.get("l1_ratio", .5), max_iter=5000))
    if name == "RandomForest": return make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=kw.get("n_estimators", 300), min_samples_leaf=kw.get("min_samples_leaf", 10), max_features=kw.get("max_features", .5), n_jobs=-1, random_state=0))
    if name == "ExtraTrees":   return make_pipeline(SimpleImputer(strategy="median"), ExtraTreesRegressor(n_estimators=kw.get("n_estimators", 300), min_samples_leaf=kw.get("min_samples_leaf", 10), max_features=kw.get("max_features", .7), n_jobs=-1, random_state=0))
    if name == "HistGradientBoosting": return HistGradientBoostingRegressor(loss="absolute_error", learning_rate=kw.get("learning_rate", .05), max_iter=kw.get("max_iter", 300), max_leaf_nodes=kw.get("max_leaf_nodes", 15), min_samples_leaf=kw.get("min_samples_leaf", 20), random_state=0)
    if name == "LightGBM":     return lgb.LGBMRegressor(objective="l1", n_estimators=kw.get("n_estimators", 300), learning_rate=kw.get("learning_rate", .03), num_leaves=kw.get("num_leaves", 15), min_child_samples=kw.get("min_child_samples", 20), subsample=.8, subsample_freq=1, colsample_bytree=.8, verbose=-1, random_state=0, n_jobs=-1)
    if name == "XGBoost":      return xgb.XGBRegressor(objective="reg:absoluteerror", n_estimators=kw.get("n_estimators", 300), learning_rate=kw.get("learning_rate", .03), max_depth=kw.get("max_depth", 4), subsample=.8, colsample_bytree=.8, min_child_weight=kw.get("min_child_weight", 5), random_state=0, n_jobs=-1)
    if name == "GradientBoosting(sklearn)":
        from sklearn.ensemble import GradientBoostingRegressor
        return make_pipeline(SimpleImputer(strategy="median"), GradientBoostingRegressor(loss="absolute_error", n_estimators=200, max_depth=3, learning_rate=.05, subsample=.8, random_state=0))

CANDIDATES = ["Ridge","ElasticNet","RandomForest","ExtraTrees","HistGradientBoosting","LightGBM","XGBoost","GradientBoosting(sklearn)"]
FEATS = prep.ALL_FEATURES
def fit_eval(name, feats=FEATS, train=tr, **kw):
    m = make_model(name, **kw); t = time.time()
    m.fit(train[feats], train.y)
    pv = va.price.values*np.exp(m.predict(va[feats])); pt = te.price.values*np.exp(m.predict(te[feats]))
    return m, metrics(va, pv), metrics(te, pt), time.time()-t

screen = []
for name in CANDIDATES:
    _, mv, mt, sec = fit_eval(name)
    screen.append({"model": name, "val_MAPE_%": mv["MAPE_%"], "val_MAE_ZMW": mv["MAE_ZMW"], "test_MAPE_%": mt["MAPE_%"], "test_MAE_ZMW": mt["MAE_ZMW"], "fit_sec": sec})
screen = pd.DataFrame(screen).sort_values("val_MAPE_%").reset_index(drop=True)
display(screen.round(3))
TOP5 = screen.model.head(5).tolist(); print("Top-5 by validation MAPE:", TOP5)

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
x = np.arange(len(screen)); w = .38
ax.bar(x-w/2, screen["val_MAPE_%"], w, label="validation", color="#2a6f97"); ax.bar(x+w/2, screen["test_MAPE_%"], w, label="test", color="#e07a5f")
ax.axhline(10, color="k", ls="--", lw=1); ax.text(len(screen)-.5, 10.1, "10 % target", ha="right", fontsize=8)
nv = baselines.loc[0]; ax.axhline(nv["val_MAPE_%"], color="#2a6f97", ls=":", lw=1); ax.axhline(nv["test_MAPE_%"], color="#e07a5f", ls=":", lw=1)
ax.set_xticks(x); ax.set_xticklabels(screen.model, rotation=30, ha="right"); ax.set_ylabel("MAPE (%)")
ax.set_title("Model screening (dotted lines = naive baseline)"); ax.legend()
save("03_model_screening.png")

## 4.4 Light hyper-parameter tuning (validation set only)
A small grid per model; test data are never used for tuning.

In [ ]:
GRIDS = {
 "Ridge": [{"alpha": a} for a in [1, 10, 100, 1000]],
 "ElasticNet": [{"alpha": a, "l1_ratio": l} for a in [1e-4, 1e-3, 1e-2] for l in [.2,.8]],
 "RandomForest": [{"min_samples_leaf": l, "max_features": f} for l in [5, 20, 50] for f in [.33, .6]],
 "ExtraTrees": [{"min_samples_leaf": l, "max_features": f} for l in [5, 20, 50] for f in [.5, .8]],
 "HistGradientBoosting": [{"learning_rate": lr, "max_leaf_nodes": n, "min_samples_leaf": s} for lr in [.03,.1] for n in [7,15] for s in [20,60]],
 "LightGBM": [{"learning_rate": lr, "num_leaves": n, "min_child_samples": s} for lr in [.02,.05] for n in [7,15,31] for s in [20,60]],
 "XGBoost": [{"learning_rate": lr, "max_depth": d, "min_child_weight": s} for lr in [.02,.05] for d in [3,5] for s in [5,20]],
 "GradientBoosting(sklearn)": [{}],
}
best_params, tuning_log = {}, []
for name in TOP5:
    best = (1e9, {})
    for p in GRIDS[name]:
        _, mv, _, _ = fit_eval(name, **p)
        tuning_log.append({"model": name, **{k:str(v) for k,v in p.items()}, "val_MAPE_%": mv["MAPE_%"]})
        if mv["MAPE_%"] < best[0]: best = (mv["MAPE_%"], p)
    best_params[name] = best[1]; print(f"{name:28s} best val MAPE {best[0]:.3f}  params {best[1]}")
pd.DataFrame(tuning_log).to_csv(os.path.join(ROOT,"reports","tuning_log.csv"), index=False)

## 4.5 Ablation study on the 5 best models

For each tuned model we re-train and re-evaluate after

* removing **one feature group at a time** (`lags`, `momentum`, `rolling`, `seasonal`, `imputation`, `series_id`, `cross_section`),
* keeping **only lags** (minimal autoregressive model),
* **dropping imputed rows from the training data** (does imputation help or hurt?).

A positive Δ (higher MAPE than the full model) means the removed component was useful.

In [ ]:
variants = {"full": (FEATS, None)}
for g, cols in prep.FEATURE_GROUPS.items():
    variants[f"- {g}"] = ([f for f in FEATS if f not in cols], None)
variants["only lags"] = (prep.FEATURE_GROUPS["lags"], None)
variants["train w/o imputed rows"] = (FEATS, "drop_imputed")

abl, store = [], {}
for name in TOP5:
    for vname, (fs, special) in variants.items():
        train = tr if special is None else tr[(tr.is_imputed_t==0) & (tr.y_imputed==0)]
        m, mv, mt, _ = fit_eval(name, feats=fs, train=train, **best_params[name])
        abl.append({"model": name, "variant": vname, "n_features": len(fs), "val_MAPE_%": mv["MAPE_%"], "val_MAE_ZMW": mv["MAE_ZMW"], "test_MAPE_%": mt["MAPE_%"], "test_MAE_ZMW": mt["MAE_ZMW"]})
        store[(name, vname)] = (m, fs)
abl = pd.DataFrame(abl)
full = abl[abl.variant=="full"].set_index("model")
abl["delta_val_MAPE"] = abl.apply(lambda r: r["val_MAPE_%"] - full.loc[r.model, "val_MAPE_%"], axis=1)
abl["delta_test_MAPE"] = abl.apply(lambda r: r["test_MAPE_%"] - full.loc[r.model, "test_MAPE_%"], axis=1)
abl.to_csv(os.path.join(ROOT,"reports","ablation_results.csv"), index=False)
display(abl.round(3))

In [ ]:
pv = abl.pivot(index="variant", columns="model", values="delta_val_MAPE").drop("full")
fig, ax = plt.subplots(figsize=(9,4.5))
im = ax.imshow(pv.values, cmap="RdBu_r", vmin=-abs(pv.values).max(), vmax=abs(pv.values).max(), aspect="auto")
ax.set_xticks(range(pv.shape[1])); ax.set_xticklabels(pv.columns, rotation=25, ha="right"); ax.set_yticks(range(pv.shape[0])); ax.set_yticklabels(pv.index)
for i in range(pv.shape[0]):
    for j in range(pv.shape[1]): ax.text(j, i, f"{pv.values[i,j]:+.2f}", ha="center", va="center", fontsize=8)
ax.grid(False); ax.set_title("Ablation: change in validation MAPE (pp) vs full model\n(red = component helps, blue = removing it improves)"); plt.colorbar(im)
save("03_ablation_heatmap.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10,4.5))
piv = abl.pivot(index="variant", columns="model", values="val_MAPE_%")
piv.plot.bar(ax=ax, width=.8); ax.axhline(10, color="k", ls="--", lw=1)
ax.set_ylabel("validation MAPE (%)"); ax.set_ylim(piv.min().min()-.7, piv.max().max()+.3); ax.set_title("Ablation – validation MAPE by variant and model"); plt.xticks(rotation=30, ha="right")
save("03_ablation_bars.png")

**Reading the ablation** – see the heatmap. Check in particular (a) whether the `imputation` group (the flag the algorithm uses to know which values were imputed) helps, (b) whether training without imputed rows is better, (c) how much the cross-province and seasonal features contribute versus plain lags.

## 4.6 Model selection
We pick the (model, variant) with the **smallest validation MAPE** that is **under 10 %**; the test set is only used to report the final unbiased figure.

In [ ]:
cand = abl.sort_values("val_MAPE_%").reset_index(drop=True)
display(cand.head(8).round(3))
under = cand[cand["val_MAPE_%"] < 10]
choice = (under if len(under) else cand).iloc[0]
MEETS_TARGET_VAL = bool(len(under))
print("CHOSEN:", choice.model, "|", choice.variant, f"| val MAPE {choice['val_MAPE_%']:.3f}  test MAPE {choice['test_MAPE_%']:.3f}  test MAE {choice['test_MAE_ZMW']:.3f} ZMW")
print("meets <10% on validation:", MEETS_TARGET_VAL, "| meets <10% on test:", choice["test_MAPE_%"] < 10)

## 4.7 Prediction intervals – normalised split-conformal
Residuals on the **validation** set (model trained on train only) are scaled by each series' recent volatility (`roll_std_6`, floored). The 90 % quantile of the scaled residuals `q` gives

`lower = price_t · exp(ŷ − q·σ)`, `upper = price_t · exp(ŷ + q·σ)`

Coverage is then checked on the untouched **test** set.

In [ ]:
best_model, FS = store[(choice.model, choice.variant)]
SIG_FLOOR = 0.03
sig_default = float(F.roll_std_6.median())
def sigma(d): return d.roll_std_6.fillna(d.roll_std_3).fillna(sig_default).clip(lower=SIG_FLOOR).values
def interval(d, model, q):
    yh = model.predict(d[FS]); s = sigma(d)
    return d.price.values*np.exp(yh), d.price.values*np.exp(yh-q*s), d.price.values*np.exp(yh+q*s)

okv = va.y_imputed.values==0
res_v = np.abs(va.y.values - best_model.predict(va[FS]))/sigma(va)
ALPHA = .10
n = okv.sum(); q = float(np.quantile(res_v[okv], min(1, np.ceil((n+1)*(1-ALPHA))/n)))
pred, lo, hi = interval(te, best_model, q); okt = te.y_imputed.values==0
cover = np.mean((te.y_price.values[okt]>=lo[okt])&(te.y_price.values[okt]<=hi[okt]))
width = np.mean((hi-lo)[okt]/pred[okt])*100
print(f"conformal q = {q:.3f} | test coverage (target 90%) = {cover*100:.1f}% | mean relative width = {width:.1f}% of forecast")

## 4.8 Refit and save

* `best_model_trainonly.joblib` – trained on the train split only (used in notebook 04 for honest test evaluation).
* `best_model.joblib` – **deployed model**, same hyper-parameters, refit on *all* rows (train+val+test) so the most recent months inform the forecast. Conformal quantile `q` is kept from the validation calibration.

In [ ]:
import joblib, copy
bundle_common = dict(features=FS, model_name=choice.model, variant=choice.variant, params=best_params[choice.model],
                     conformal_q=q, conformal_alpha=ALPHA, sigma_floor=SIG_FLOOR, sigma_default=sig_default,
                     province_codes=codes["province_codes"], commodity_codes=codes["commodity_codes"])
joblib.dump({**bundle_common, "model": best_model}, os.path.join(MODELS, "best_model_trainonly.joblib"))

train_all = F if choice.variant != "train w/o imputed rows" else F[(F.is_imputed_t==0)&(F.y_imputed==0)]
final = make_model(choice.model, **best_params[choice.model]).fit(train_all[FS], train_all.y)
joblib.dump({**bundle_common, "model": final, "trained_through": str(F.month.max().date())}, os.path.join(MODELS, "best_model.joblib"))
meta = {"model": choice.model, "variant": choice.variant, "params": {k:str(v) for k,v in best_params[choice.model].items()},
        "features": FS, "validation_MAPE_pct": float(choice["val_MAPE_%"]), "test_MAPE_pct": float(choice["test_MAPE_%"]),
        "test_MAE_ZMW": float(choice["test_MAE_ZMW"]), "test_MdAPE_pct": float(metrics(te, pred)["MdAPE_%"]),
        "naive_test_MAPE_pct": float(baselines.loc[0,"test_MAPE_%"]), "naive_val_MAPE_pct": float(baselines.loc[0,"val_MAPE_%"]),
        "meets_10pct_target_validation": bool(choice["val_MAPE_%"]<10), "meets_10pct_target_test": bool(choice["test_MAPE_%"]<10),
        "interval_coverage_test": float(cover), "interval_nominal": 1-ALPHA, "interval_mean_rel_width_pct": float(width),
        "n_series": int(F.series_id.nunique()), "n_training_rows": int(len(train_all))}
json.dump(meta, open(os.path.join(MODELS, "metadata.json"), "w"), indent=2); print(json.dumps(meta, indent=2))
screen.to_csv(os.path.join(ROOT,"reports","model_screening.csv"), index=False); baselines.to_csv(os.path.join(ROOT,"reports","baselines.csv"), index=False)